# 01 - Training driver (Colab GPU)

Trains one (variant, task, fraction, seed) point at a time and exports what notebooks 02 and 03 need. Runs with `done.json` are skipped, runs with `failed.json` are retried, interrupted runs restart; order is cheap-first. `STOP_AFTER_HOURS` stops starting new runs after that wall time. Set `SMOKE_TEST = True` once on a fresh setup.

Per run, `<root>/runs/<variant>/<task>/f<NNNN>/seed<k>/` gets `history.json`, `metrics.json`, `closure.json`, `r_val.npz`, `r_toys_mu0p4.npy`, `r_toys_mu2p5.npy` and `done.json` (or `failed.json`); `final.pt` only with `keep_checkpoint`.

In [ ]:
# --- Colab setup (run once per session) ---
import os, sys
IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    !test -d /content/EveNet-Lite || git clone -q https://github.com/Dekiofsa/EveNet-Lite.git /content/EveNet-Lite
    !pip install -q --no-deps evenet-core==0.3.0
    !pip install -q --no-deps -e /content/EveNet-Lite
    !pip install -q lightning scipy scikit-learn wandb huggingface-hub pyyaml rich opt-einsum pyarrow tqdm regex
    sys.path.insert(0, "/content/EveNet-Lite")
def _find_scan_dir():
    """Locate the folder holding nsbi_scan.py on Drive (any folder name works)."""
    import glob as _glob
    cands = ["/content/drive/MyDrive/Research/NSBI/datascan",
             "/content/drive/MyDrive/Research/NSBI/Training Statistics Scan"]
    for pat in ("/content/drive/MyDrive/Research/NSBI/*/nsbi_scan.py",
                "/content/drive/MyDrive/*/nsbi_scan.py",
                "/content/drive/MyDrive/*/*/nsbi_scan.py"):
        if any(os.path.isfile(os.path.join(c, "nsbi_scan.py")) for c in cands):
            break
        cands += sorted({os.path.dirname(p) for p in _glob.glob(pat)})
    for c in cands:
        if os.path.isfile(os.path.join(c, "nsbi_scan.py")):
            return c
    raise FileNotFoundError("nsbi_scan.py not found on Drive: copy the whole scan folder "
                            "(nsbi_scan.py + notebooks) under MyDrive/Research/NSBI/ and rerun this cell")
SCAN_DIR = _find_scan_dir() if IN_COLAB else os.path.dirname(os.path.abspath("__file__"))
sys.path.insert(0, SCAN_DIR)
import nsbi_scan as ns
print("nsbi_scan from", ns.__file__)

In [ ]:
# ===================== CONFIG - the only cell you normally edit =====================
TIER = "tier1"
DEVICE = "cuda"
STOP_AFTER_HOURS = 9.0
MAX_RUNS = None
SMOKE_TEST = False
OVERRIDES = {"wandb_project": "nsbi-evenet-datascan"}
                           # {"num_workers": 2, "keep_checkpoint": False, "wandb_project": "nsbi-scan"}

In [ ]:
import dataclasses, glob, json, math, time
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = ns.resolve_root(SCAN_DIR)
cfg = ns.ScanConfig.load(ROOT).with_root(ROOT)   # scan_config.json written by notebook 00
cfg = dataclasses.replace(cfg, **OVERRIDES)

steps_per_epoch = math.ceil(cfg.epoch_size / cfg.batch_size)
print("root             ", cfg.root)
print("fractions        ", cfg.fractions)
print("variants / seeds ", cfg.variants, "/", cfg.seeds, "| tasks", cfg.tasks)
print("step budget      ", f"batch {cfg.batch_size} x {cfg.epoch_size} rows per epoch = {steps_per_epoch} steps/epoch, "
      f"max {cfg.max_epochs} epochs, early-stop patience {cfg.early_stop_patience}, val monitor {cfg.val_monitor_rows} rows")
print("scratch recipe   ", f"lr {cfg.scratch_lr}, weight_decay {cfg.scratch_weight_decay}")
print("housekeeping     ", f"num_workers {cfg.num_workers} | keep_checkpoint {cfg.keep_checkpoint} | wandb_project {cfg.wandb_project}")
print("measurement      ", f"calib ref {cfg.calib_ref_events} bkg_train events | lumi {cfg.lumi} fb^-1 | mu_true {cfg.mu_true_values}")

needed = ([f"{p}_{s}.npz" for p in ns.PROCESSES for s in ("train", "val")] + ["summary.json", "xs.json"]
          + [f"toys_{ns.MU_TAGS[m]}.npz" for m in cfg.mu_true_values])
missing = [f for f in needed if not (cfg.data_dir / f).exists()]
if missing:
    print("data cache       ", f"INCOMPLETE under {cfg.data_dir}: {missing} -> run notebook 00 before training")
else:
    print("data cache       ", "complete under", cfg.data_dir)

try:
    import torch
    gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"
    print("torch            ", torch.__version__, "| cuda available:", torch.cuda.is_available(), "| device:", gpu)
except ImportError:
    print("torch not installed: the status cells work, the training cells will not")

In [ ]:
if cfg.wandb_project:
    import wandb
    wandb.login()
    print("W&B project:", cfg.wandb_project, "| entity:", cfg.wandb_entity)
else:
    print("W&B disabled (cfg.wandb_project is None). Set OVERRIDES['wandb_project'] in CONFIG to enable it.")

In [ ]:
TIERS = {
    "tier1": ns.run_matrix(cfg, variants=["scratch", "pretrained"], seeds=[0]),
    "tier2": ns.run_matrix(cfg, variants=["scratch", "pretrained"], fractions=[0.01, 0.03, 0.10], seeds=[1, 2])
             + ns.run_matrix(cfg, variants=["frozen"], seeds=[0]),
    "tier3": ns.run_matrix(cfg, variants=["scratch", "pretrained"], fractions=[0.30, 1.0], seeds=[1, 2]),
}
TIERS["all"] = TIERS["tier1"] + TIERS["tier2"] + TIERS["tier3"]

def cheap_first(spec):
    return (spec["frac"], spec["seed"], ns.VARIANTS.index(spec["variant"]), ns.TASKS.index(spec["task"]))

specs = sorted(TIERS[TIER], key=cheap_first)
print("runs per tier:", {k: len(v) for k, v in TIERS.items()})
print(f"selected TIER = {TIER!r}: {len(specs)} runs, first = {specs[0]}, last = {specs[-1]}")
display(pd.DataFrame(specs).groupby(["variant", "frac"]).size().unstack("frac", fill_value=0))
print("training events per hypothesis at each fraction:")
display(ns.training_event_table(cfg))

In [ ]:
status = ns.scan_status(cfg, specs)
print(status["status"].value_counts().to_string())
display(status.pivot_table(index=["variant", "task", "seed"], columns="frac", values="status", aggfunc="first"))

In [ ]:
if SMOKE_TEST:
    import torch
    assert not DEVICE.startswith("cuda") or torch.cuda.is_available(), (
        "no CUDA device: change the runtime type (Runtime > Change runtime type > GPU) or set DEVICE='auto'")
    assert not missing, f"data cache incomplete under {cfg.data_dir}: {missing} -> run notebook 00 first"
    smoke = dict(variant="pretrained", task="sig_over_bkg", frac=min(cfg.fractions), seed=0)
    m = ns.train_one(cfg, device=DEVICE, **smoke)
    rd = ns.run_dir(cfg, **smoke)
    ok = m is not None and (rd / "done.json").exists() and (rd / "metrics.json").exists()
    print("smoke test", "PASSED" if ok else "FAILED", "->", rd.relative_to(cfg.runs_dir).as_posix())
    if m:
        print({k: m.get(k) for k in ("epochs_run", "best_epoch", "val_auc_full", "C_ref", "closure_chi2_ndf", "fit_seconds")})
    assert ok, "see failed.json (printed by the 'Failed runs' cell below) and fix nsbi_scan.py before launching a tier"
else:
    print("smoke test skipped (SMOKE_TEST = False)")

In [ ]:
import torch
assert not DEVICE.startswith("cuda") or torch.cuda.is_available(), (
    "no CUDA device: change the runtime type (Runtime > Change runtime type > GPU) or set DEVICE='auto'")
assert not missing, f"data cache incomplete under {cfg.data_dir}: {missing} -> run notebook 00 first"
STOP_AFTER = STOP_AFTER_HOURS * 3600.0
t0 = time.time()
result = ns.run_driver(cfg, specs, device=DEVICE, max_runs=MAX_RUNS, stop_after_seconds=STOP_AFTER)
print(f"session wall time: {(time.time() - t0) / 3600:.2f} h")
print(result["status"].value_counts().to_string())
display(result)

In [ ]:
status = ns.scan_status(cfg, specs)
print(status["status"].value_counts().to_string())
display(status.pivot_table(index=["variant", "task", "seed"], columns="frac", values="status", aggfunc="first"))

In [ ]:
import matplotlib.pyplot as plt

def _load_json(path):
    try:
        return json.load(open(path))
    except (ValueError, OSError) as e:
        print(Path(path).parent.relative_to(cfg.runs_dir).as_posix(), Path(path).name, "unreadable:", e)
        return None

hist_files = glob.glob(str(cfg.runs_dir / "*" / "*" / "f*" / "seed*" / "history.json"))
if not hist_files:
    print("no history.json found yet under", cfg.runs_dir)
else:
    latest = max(hist_files, key=os.path.getmtime)
    rd = Path(latest).parent
    label = rd.relative_to(cfg.runs_dir).as_posix()
    hist = pd.DataFrame(_load_json(latest) or [])
    if "val_loss" not in hist or not hist["val_loss"].notna().any():
        print(label, "has no val_loss rows yet")
        display(hist)
    else:
        best = int(hist["val_loss"].idxmin())
        fig, ax = plt.subplots(figsize=(6, 3.4))
        if "train_loss" in hist:
            ax.plot(hist["epoch"], hist["train_loss"], color="#2a78d6", lw=1.5, label="train (uniform, weighted batches)")
        ax.plot(hist["epoch"], hist["val_loss"], color="#eb6834", lw=1.5, marker="o", ms=3, label="val (weighted, monitor subset)")
        ax.axvline(hist.loc[best, "epoch"], color="#52514e", lw=1, ls="--")
        ax.annotate(f"best epoch {int(hist.loc[best, 'epoch'])}", (hist.loc[best, "epoch"], hist.loc[best, "val_loss"]),
                    xytext=(6, 8), textcoords="offset points", fontsize=8, color="#52514e")
        ax.set_xlabel(f"epoch ({steps_per_epoch} optimizer steps each)")
        ax.set_ylabel("cross-entropy (train: uniform on weighted batches; val: weighted)")
        ax.set_title(f"most recent run: {label}", fontsize=10)
        ax.grid(alpha=0.25)
        ax.legend(frameon=False)
        for side in ("top", "right"):
            ax.spines[side].set_visible(False)
        plt.tight_layout()
        plt.show()
        cols = [c for c in ("epoch", "global_step", "train_loss", "val_loss", "elapsed_s") if c in hist]
        display(hist[cols].round(5))
    if not (rd / "metrics.json").exists():
        print(label, "has no metrics.json yet (partial or failed run)")
    else:
        m = _load_json(rd / "metrics.json")   # None if train_one crashed while writing it
        if m is not None:
            keys = ["epochs_run", "best_epoch", "best_step", "best_val_loss_monitor", "val_bce_full", "val_auc_full",
                    "C_ref", "C_sub", "C_val", "closure_chi2_ndf", "n_trainable_params", "fit_seconds", "total_seconds"]
            display(pd.Series({k: m.get(k) for k in keys}, name=label, dtype=object).to_frame())

In [ ]:
# Failed runs and their recorded error (full traceback in failed.json)
failed = status[status["status"] == "failed"]
if failed.empty:
    print("no failed runs")
for _, s in failed.iterrows():
    rd = ns.run_dir(cfg, s["variant"], s["task"], float(s["frac"]), int(s["seed"]))
    err = json.load(open(rd / "failed.json"))
    print("=" * 100)
    print(rd.relative_to(cfg.runs_dir).as_posix(), "|", err.get("time"), "|", err.get("error"))
    print(err.get("traceback", "")[-2000:])
if not failed.empty:
    print("Failed runs are retried by the next run_driver launch (failed.json is removed when the retry starts).")

## Next

Notebook 02 measures mu from the exported ratios without a GPU; set `NSBI_SCAN_ROOT` to a synced copy of the tree to run it locally.